In [0]:
%sql
DROP TABLE IF EXISTS demo.bronze.customer;
CREATE TABLE demo.bronze.customer (
  index	INT,
  customer_id	STRING,
  first_name	STRING,
  last_name	STRING,
  company	STRING,
  city	STRING,
  country	STRING,
  phone1	STRING,
  phone2	STRING,
  email	STRING,
  subscription_date DATE,
  website STRING,
  CONSTRAINT customer_pk PRIMARY KEY (customer_id)

)
USING DELTA;

DROP TABLE IF EXISTS demo.bronze.product;
CREATE TABLE demo.bronze.product (
  product_id INT,
  name STRING,
  description STRING,
  brand STRING,
  category STRING,
  price DOUBLE,
  currency STRING,
  stock INT,
  ean LONG,
  color STRING,
  size STRING,
  availability STRING,
  internal_id INT
)
USING DELTA;

DROP TABLE IF EXISTS demo.bronze.order;
CREATE TABLE demo.bronze.order (
  order_id INT,
  customer_id STRING,
  product_id INT,
  order_amount DOUBLE,
  order_date DATE,
  order_quantity INT,
  order_status STRING,
  order_channel STRING,
  ship_date DATE,
  ship_mode STRING,
  ship_cost DOUBLE,
  remark STRING
)
USING DELTA;

In [0]:
%python
from pyspark.sql.functions import md5, concat_ws, col

df_customer = spark.read.option("header", "true").option("inferSchema", "true").csv("/Volumes/demo/bronze/data/customers-10000.csv")

df_customer = df_customer.withColumnRenamed("Index", "index") \
    .withColumnRenamed("Customer Id", "customer_id") \
    .withColumnRenamed("First Name", 'first_name') \
    .withColumnRenamed("Last Name", 'last_name') \
    .withColumnRenamed("Company", "company") \
    .withColumnRenamed("City", "city") \
    .withColumnRenamed("Country", "country") \
    .withColumnRenamed("Phone 1", "phone1") \
    .withColumnRenamed("Phone 2", "phone2") \
    .withColumnRenamed("Email", "email") \
    .withColumnRenamed("Subscription Date", "subscription_date") \
    .withColumnRenamed("Website", "website")

df_customer.write.format("delta").mode("overwrite").saveAsTable("demo.bronze.customer")


In [0]:
from pyspark.sql.functions import md5, concat_ws, col, monotonically_increasing_id
df_product = spark.read.option("header", "true").option("inferSchema", "true").csv("/Volumes/demo/bronze/data/products-10000.csv")

df_product = df_product.withColumn("product_id", monotonically_increasing_id().cast("int")) \
    .withColumnRenamed("Name", "name") \
    .withColumnRenamed("Description", "description") \
    .withColumnRenamed("Brand", "brand") \
    .withColumnRenamed("Category", "category") \
    .withColumnRenamed("Price", "price").withColumn("price", col("price").cast("double")) \
    .withColumnRenamed("Currency", "currency") \
    .withColumnRenamed("Stock", "stock") \
    .withColumnRenamed("EAN", "ean") \
    .withColumnRenamed("Color", "color") \
    .withColumnRenamed("Size", "size") \
    .withColumnRenamed("Availability", "availability") \
    .withColumnRenamed("Internal Id", "internal_id") \
    .select("product_id", "name", "description", "brand", "category", "price", "currency", "stock", "ean", "color", "size", "availability", "internal_id")

df_product.write.format("delta").mode("overwrite").saveAsTable("demo.bronze.product")


In [0]:
from pyspark.sql.functions import monotonically_increasing_id, lit, current_date, rand, when

df_customer = spark.table("demo.bronze.customer")
df_product = spark.table("demo.bronze.product")

df_customer_with_id = df_customer.withColumn("id", monotonically_increasing_id())
df_product_with_id = df_product.withColumn("id", monotonically_increasing_id())

# Join on the ID and drop it
merged_df = df_customer_with_id.join(df_product_with_id, "id").drop("id")

df_order = merged_df.withColumn("order_id", monotonically_increasing_id().cast("int")) \
    .withColumn("order_date", current_date()) \
    .withColumn("order_quantity", (rand(seed=42) * 24).cast("int")) \
    .withColumn("order_status",  \
                when(rand() < 0.33, "created") \
                .when(rand() < 0.66, "shipped") \
                .otherwise("delivered")) \
    .withColumn("order_channel", lit("online")) \
    .withColumn("ship_date", current_date()) \
    .withColumn("ship_mode", lit("standard")) \
    .withColumn("ship_cost", lit(5.0)) \
    .withColumn("remark", lit("auto-generated")) \
    .withColumn("order_amount", col("price") * col("order_quantity")) \
    .select(
        "order_id",
        "customer_id",
        "product_id",
        "order_amount",
        "order_date",
        "order_quantity",
        "order_status",
        "order_channel",
        "ship_date",
        "ship_mode",
        "ship_cost",
        "remark"
    )

df_order.write.format("delta").mode("overwrite").saveAsTable("demo.bronze.order")


In [0]:
%sql
-- Hub: h_customer
DROP TABLE IF EXISTS demo.silver.h_customer;
CREATE TABLE demo.silver.h_customer (
  customer_hk STRING NOT NULL,
  customer_id STRING NOT NULL,
  load_time TIMESTAMP,
  record_source STRING,
  CONSTRAINT h_customer_pk PRIMARY KEY (customer_hk)
)
USING DELTA;

-- Satellite: s_customer
DROP TABLE IF EXISTS demo.silver.s_customer;
CREATE TABLE demo.silver.s_customer (
  customer_hk STRING NOT NULL,
  first_name STRING,
  last_name STRING,
  company STRING,
  city STRING,
  country STRING,
  phone1 STRING,
  phone2 STRING,
  email STRING,
  subscription_date DATE,
  website STRING,
  value_hash STRING,
  load_time TIMESTAMP,
  record_source STRING,
  CONSTRAINT s_customer_pk PRIMARY KEY (customer_hk)
)
USING DELTA;

-- Hub: h_product
DROP TABLE IF EXISTS demo.silver.h_product;
CREATE TABLE demo.silver.h_product (
  product_hk STRING NOT NULL,
  product_id INT NOT NULL,
  load_time TIMESTAMP,
  record_source STRING,
  CONSTRAINT h_product_pk PRIMARY KEY (product_hk)
)
USING DELTA;

-- Satellite: s_product
DROP TABLE IF EXISTS demo.silver.s_product;
CREATE TABLE demo.silver.s_product (
  product_hk STRING NOT NULL,
  name STRING,
  description STRING,
  brand STRING,
  category STRING,
  price DOUBLE,
  currency STRING,
  stock INT,
  ean LONG,
  color STRING,
  size STRING,
  availability STRING,
  internal_id INT,
  value_hash STRING,
  load_time TIMESTAMP,
  record_source STRING,
  CONSTRAINT s_product_pk PRIMARY KEY (product_hk)
)
USING DELTA;

-- Link: l_order
DROP TABLE IF EXISTS demo.silver.l_order;
CREATE TABLE demo.silver.l_order (
  order_hk STRING NOT NULL,
  order_id INT NOT NULL,
  customer_id STRING NOT NULL,
  product_id INT NOT NULL,
  load_time TIMESTAMP,
  record_source STRING,
  CONSTRAINT l_order_pk PRIMARY KEY (order_hk)
)
USING DELTA;

-- Satellite: s_l_order
DROP TABLE IF EXISTS demo.silver.s_l_order;
CREATE TABLE demo.silver.s_l_order (
  order_hk STRING NOT NULL,
  order_amount DOUBLE,
  order_date DATE,
  order_quantity INT,
  order_status STRING,
  order_channel STRING,
  ship_date DATE,
  ship_mode STRING,
  ship_cost DOUBLE,
  remark STRING,
  value_hash STRING,
  load_time TIMESTAMP,
  record_source STRING,
  CONSTRAINT s_l_order_pk PRIMARY KEY (order_hk)
)
USING DELTA;

In [0]:
from pyspark.sql.functions import md5, concat_ws, col, sha2

def gen_customer_hk(df):
  df = df.withColumn(
    'customer_hk', 
    md5(
      col('customer_id')
      )
    )
  return df

def gen_customer_value_hash(df):
  df = df.withColumn(
    "value_hash", 
    sha2(
      concat_ws(
        "||", 
        col("first_name"), 
        col("last_name"), 
        col("company"), 
        col("city"), 
        col("country"), 
        col("phone1"), 
        col("phone2"), 
        col("email"),
        col("subscription_date"), 
        col("website")
        ), 
      256 
      )
    )
  return df

def gen_product_hk(df):
  df = df.withColumn('product_hk', md5(col('product_id').cast('string')))
  return df

def gen_product_value_hash(df):
  df = df.withColumn(
    "value_hash",
    sha2(
      concat_ws(
        "||",
        col("name"),
        col("description"),
        col("brand"),
        col("category"),
        col("price"),
        col("currency"),
        col("stock"),
        col("ean"),
        col("color"),
        col("size"),
        col("availability"),
        col("internal_id")
      ),
      256
    )
  )
  return df

def gen_order_hk(df):
  df = df.withColumn(
    'order_hk', 
    md5(concat_ws(
      '||', 
      col('order_id').cast('string'), 
      col('customer_id'), 
      col('product_id').cast('string')
    ))
  )
  return df

def gen_order_value_hash(df):
  df = df.withColumn(
    "value_hash",
    sha2(
      concat_ws(
        "||",
        col("order_amount").cast("string"),
        col("order_date"),
        col("order_quantity").cast("string"),
        col("order_status"),
        col("order_channel"),
        col("ship_date"),
        col("ship_mode"),
        col("ship_cost").cast("string"),
        col("remark")
      ),
      256
    )
  )
  return df



In [0]:
import pandas as pd
from datetime import datetime
from pyspark.sql.functions import lit

df_h_cust = spark.table("demo.bronze.customer")
df_h_cust = gen_customer_hk(df_h_cust)
df_h_cust = df_h_cust.select("customer_hk", "customer_id")
df_h_cust = df_h_cust.withColumn("load_time", lit(datetime.now()))
df_h_cust = df_h_cust.withColumn("record_source", lit("products-10000.csv"))
df_h_cust.write.format("delta").mode("overwrite").saveAsTable("demo.silver.h_customer")

# Satellite : s_customer
df_s_cust = spark.table("demo.bronze.customer")
df_s_cust = gen_customer_hk(df_s_cust)
df_s_cust = gen_customer_value_hash(df_s_cust)
df_s_cust = df_s_cust.withColumn("load_time", lit(datetime.now())) \
                     .withColumn("record_source", lit("products-10000.csv"))
df_s_cust = df_s_cust.select(
    "customer_hk",
    "first_name",
    "last_name",
    "company",
    "city",
    "country",
    "phone1",
    "phone2",
    "email",
    "subscription_date",
    "website",
    "value_hash",
    "load_time",
    "record_source"
)
df_s_cust.write.format("delta").mode("overwrite").saveAsTable("demo.silver.s_customer")




In [0]:
from datetime import datetime
from pyspark.sql.functions import lit

# Hub: h_product
df_h_prod = spark.table("demo.bronze.product")
df_h_prod = gen_product_hk(df_h_prod)
df_h_prod = df_h_prod.select("product_hk", "product_id")
df_h_prod = df_h_prod.withColumn("load_time", lit(datetime.now()))
df_h_prod = df_h_prod.withColumn("record_source", lit("products-10000.csv"))
df_h_prod.write.format("delta").mode("overwrite").saveAsTable("demo.silver.h_product")

# Satellite: s_product
df_s_prod = spark.table("demo.bronze.product")
df_s_prod = gen_product_hk(df_s_prod)
df_s_prod = gen_product_value_hash(df_s_prod)
df_s_prod = df_s_prod.withColumn("load_time", lit(datetime.now())) \
                     .withColumn("record_source", lit("products-10000.csv"))
df_s_prod = df_s_prod.select(
    "product_hk",
    "name",
    "description",
    "brand",
    "category",
    "price",
    "currency",
    "stock",
    "ean",
    "color",
    "size",
    "availability",
    "internal_id",
    "value_hash",
    "load_time",
    "record_source"
)
df_s_prod.write.format("delta").mode("overwrite").saveAsTable("demo.silver.s_product")

In [0]:
from datetime import datetime
from pyspark.sql.functions import lit

# Hub: h_order
df_h_order = spark.table("demo.bronze.order")
df_h_order = gen_order_hk(df_h_order)

# Link: l_order
df_l_order = df_h_order.select("order_hk", "order_id", "customer_id", "product_id")
df_l_order = df_l_order.withColumn("load_time", lit(datetime.now()))
df_l_order = df_l_order.withColumn("record_source", lit("auto-generated"))
df_l_order.write.format("delta").mode("overwrite").saveAsTable("demo.silver.l_order")

# Satellite: s_l_order
df_s_l_order = df_h_order \
    .withColumn("load_time", lit(datetime.now())) \
    .withColumn("record_source", lit("auto-generated"))

df_s_l_order = gen_order_value_hash(df_s_l_order)

df_s_l_order = df_s_l_order.select(
    "order_hk",
    "order_amount",
    "order_date",
    "order_quantity",
    "order_status",
    "order_channel",
    "ship_date",
    "ship_mode",
    "ship_cost",
    "remark",
    "value_hash",
    "load_time",
    "record_source"
)

df_s_l_order.write.format("delta").mode("overwrite").saveAsTable("demo.silver.s_l_order")

In [0]:
%sql
-- Create fact_order table
DROP TABLE IF EXISTS demo.gold.fact_order;
CREATE TABLE demo.gold.fact_order (
  order_id INT NOT NULL,
  customer_id STRING NOT NULL,
  product_id INT NOT NULL,
  order_date DATE,
  order_amount DOUBLE,
  order_status STRING,
  
  remark STRING,
  load_time TIMESTAMP,
  record_source STRING,
  CONSTRAINT fact_order_pk PRIMARY KEY (order_id)
)
USING DELTA;

-- Create dim_customer table
DROP TABLE IF EXISTS demo.gold.dim_customer;
CREATE TABLE demo.gold.dim_customer (
  customer_id STRING,
  first_name STRING,
  last_name STRING,
  company STRING,
  city STRING,
  country STRING,
  phone1 STRING,
  phone2 STRING,
  email STRING,
  subscription_date DATE,
  website STRING,
  load_time TIMESTAMP,
  record_source STRING,
  CONSTRAINT dim_customer_pk PRIMARY KEY (customer_id)
)
USING DELTA;

-- Create dim_product table
DROP TABLE IF EXISTS demo.gold.dim_product;
CREATE TABLE demo.gold.dim_product (
  product_id INT,
  name STRING,
  description STRING,
  brand STRING,
  category STRING,
  price DOUBLE,
  currency STRING,
  stock INT,
  ean LONG,
  color STRING,
  size STRING,
  availability STRING,
  internal_id INT,
  load_time TIMESTAMP,
  record_source STRING,
  CONSTRAINT dim_product_pk PRIMARY KEY (product_id)
)
USING DELTA;


In [0]:
from pyspark.sql.functions import col

df_l_order = spark.table("demo.silver.l_order").alias("lo")
df_s_l_order = spark.table("demo.silver.s_l_order").alias("slo")

df_fact_order = df_l_order.join(
    df_s_l_order,
    on="order_hk",
    how="inner"
).select(
    col("order_id"),
    col("customer_id"),
    col("product_id"),
    col("order_date"),
    col("order_amount"),
    col("order_status"),
    col("remark"),
    col("lo.load_time"),
    col("lo.record_source")
)

df_fact_order.write.format("delta").mode("overwrite").saveAsTable("demo.gold.fact_order")

df_h_customer = spark.table("demo.silver.h_customer").alias("hc")
df_s_customer = spark.table("demo.silver.s_customer").alias("sc")

df_dim_customer = df_h_customer.join(
    df_s_customer,
    on="customer_hk",
    how="inner"
).select(
    col("customer_id"),
    col("first_name"),
    col("last_name"),
    col("company"),
    col("city"),
    col("country"),
    col("phone1"),
    col("phone2"),
    col("email"),
    col("subscription_date"),
    col("website"),
    col("sc.load_time"),
    col("sc.record_source")
)

df_dim_customer.write.format("delta").mode("overwrite").saveAsTable("demo.gold.dim_customer")

# load data in table dim_product
df_h_product = spark.table("demo.silver.h_product").alias("hp")
df_s_product = spark.table("demo.silver.s_product").alias("sp")

df_dim_product = df_h_product.join(
    df_s_product,
    on="product_hk",
    how="inner"
).select(
    col("product_id"),
    col("name"),
    col("description"),
    col("brand"),
    col("category"),
    col("price"),
    col("currency"),
    col("stock"),
    col("ean"),
    col("color"),
    col("size"),
    col("availability"),
    col("internal_id"),
    col("sp.load_time"),
    col("sp.record_source")
)

df_dim_product.write.format("delta").mode("overwrite").saveAsTable("demo.gold.dim_product")

In [0]:
%sql


SELECT
  o.order_id,
  c.first_name AS customer_first_name,
  c.last_name AS customer_last_name,
  c.city AS customer_city,
  p.name AS product_name,
  p.brand AS product_brand,
  o.order_amount
FROM demo.gold.fact_order o
JOIN demo.gold.dim_customer c ON o.customer_id = c.customer_id
JOIN demo.gold.dim_product p ON o.product_id = p.product_id
ORDER BY o.order_amount DESC
LIMIT 100

order_id,customer_first_name,customer_last_name,customer_city,product_name,product_brand,order_amount
7462,Madeline,Lawson,North Mercedes,Smart Bicycle Microphone Touch,Coleman PLC,999.0
6603,Keith,Johnson,Port Kristinastad,Portable Cooler,"Mejia, Miranda and Mullins",999.0
5958,Jonathan,Powell,Odonnelltown,Fast Cooker Cooler Smart,Vincent and Sons,999.0
8348,Shaun,Beck,Lake Leonardstad,Premium Toaster Scooter Edge,Newman-Fry,999.0
2717,Gregg,Garner,Clarkeside,Mini Brush,Lawson and Sons,999.0
5861,Bonnie,Graham,Quinnbury,Fast Keyboard Lite Fast Air,"Goodwin, Webb and Schroeder",999.0
5120,Xavier,Walsh,Priscillabury,Clean Grill Charger Headphones Sense Digital,Hopkins-Lucas,999.0
6573,Dale,Benson,Shariview,Mini Oven Drone Mini Portable,Garrett Ltd,999.0
9832,Ian,Donovan,New Kristinaborough,Ultra Monitor Mini Fast,Klein PLC,999.0
1115,Sandy,Jennings,Kennedymouth,Portable Mixer Vacuum,Villa-Yates,999.0
